# MonIA Speech Performance
Candidate-only audio-driven Lucas speech performance. Input is a generated silent video plus exact V16 audio.


In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg git
!git clone --depth 1 https://github.com/TMElyralab/MuseTalk.git /kaggle/working/MuseTalk
%cd /kaggle/working/MuseTalk
%pip -q install -r requirements.txt
%pip -q install -U openmim opencv-python-headless
!mim install -q mmengine
!mim install -q 'mmcv==2.0.1'
!mim install -q 'mmdet==3.1.0'
!mim install -q 'mmpose==1.1.0'
!sh ./download_weights.sh
print('✅ MuseTalk 1.5 environment ready')


In [ ]:
import json, os, subprocess, requests, shutil
from pathlib import Path
job_path=Path('/kaggle/working/job.json')
job=json.loads(job_path.read_text(encoding='utf-8'))
assert job.get('candidateOnly') is True
assert job.get('canonicalPromotion') is False
assert job.get('engine') == 'musetalk-v1.5'
job_id=job['id']
root=Path('/kaggle/working/monia-speech')/job_id
root.mkdir(parents=True,exist_ok=True)
def dl(url,target):
    r=requests.get(url,timeout=180); r.raise_for_status(); Path(target).write_bytes(r.content); return str(target)
video=dl(job['videoUrl'],root/'input.mp4')
audio=dl(job['audioUrl'],root/'voice.wav')
normalized=str(root/'input-25fps.mp4')
subprocess.run(['ffmpeg','-y','-i',video,'-an','-vf','fps=25','-c:v','libx264','-preset','fast','-crf','18',normalized],check=True)
cfg=Path('/kaggle/working/MuseTalk/configs/inference/monia-runtime.yaml')
cfg.write_text(f'task_0:\n video_path: "{normalized}"\n audio_path: "{audio}"\n',encoding='utf-8')
result_dir=root/'musetalk-results'
cmd=['python','-m','scripts.inference','--inference_config',str(cfg),'--result_dir',str(result_dir),'--unet_model_path','models/musetalkV15/unet.pth','--unet_config','models/musetalkV15/musetalk.json','--version','v15','--ffmpeg_path','/usr/bin']
subprocess.run(cmd,cwd='/kaggle/working/MuseTalk',check=True)
candidates=sorted(result_dir.rglob('*.mp4'),key=lambda p:p.stat().st_mtime)
assert candidates,'MuseTalk produced no mp4'
final=root/'speech-synced.mp4'
shutil.copy2(candidates[-1],final)
print('✅ MuseTalk speech performance ready:',final)


In [ ]:
import json, subprocess, requests
from pathlib import Path
root=Path('/kaggle/working/monia-speech')/job['id']
gate=root/'av-coherence.json'
script=root/'monia_av_coherence.py'
script.write_bytes(requests.get('https://raw.githubusercontent.com/vartcom38-collab/marion-lucas-game/main/scripts/monia_av_coherence.py',timeout=60).content)
subprocess.run(['python',str(script),'--video',str(root/'speech-synced.mp4'),'--audio',str(root/'voice.wav'),'--report',str(gate)],check=True)
report=json.loads(gate.read_text(encoding='utf-8'))
result={
 'jobId':job['id'],'state':'candidate','candidateOnly':True,'canonicalPromotion':False,
 'engine':'musetalk-v1.5','audioAuthority':'lucas-v16','video':'speech-synced.mp4',
 'avGate':report,'phonemePerfectClaim':False
}
(root/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
print('✅ Speech-performance candidate passed MonIA AV gate')
